In [13]:
from minio import Minio
import fastavro
from io import BytesIO
import pandas as pd

# Connect to MinIO
client = Minio(
    "minio.ducdh.com",  # e.g., "minio.example.com:9000"
    access_key="minio",
    secret_key="minio123",
    secure=False,  # True if using HTTPS
)

# Download Avro file into memory
obj = client.get_object(
    "stream-bucket",
    "topics/homecredit.record.application/partition=0/homecredit.record.application+0+0000000000.avro",
)
data = BytesIO(obj.read())
obj.close()
obj.release_conn()

# Read Avro into a list of records
records = list(fastavro.reader(data))

# Create DataFrame
df = pd.DataFrame(records)

# Show first few rows
print(df.columns)

Index(['before', 'after', 'source', 'op', 'ts_ms', 'transaction'], dtype='object')


In [17]:
df["after"]

0       {'sk_id_curr': 100001, 'name_contract_type': '...
1       {'sk_id_curr': 100005, 'name_contract_type': '...
2       {'sk_id_curr': 100013, 'name_contract_type': '...
3       {'sk_id_curr': 100028, 'name_contract_type': '...
4       {'sk_id_curr': 100038, 'name_contract_type': '...
                              ...                        
9995    {'sk_id_curr': 172551, 'name_contract_type': '...
9996    {'sk_id_curr': 172556, 'name_contract_type': '...
9997    {'sk_id_curr': 172562, 'name_contract_type': '...
9998    {'sk_id_curr': 172570, 'name_contract_type': '...
9999    {'sk_id_curr': 172574, 'name_contract_type': '...
Name: after, Length: 10000, dtype: object

In [10]:
# Expand the dict in 'after' into new DataFrame columns
df_after_expanded = pd.json_normalize(df["after"])

# Optional: combine with original df if you want to keep other metadata
df_new = pd.concat([df.drop(columns=["after"]), df_after_expanded], axis=1)

print(df_new.head())

  before                                             source op          ts_ms  \
0   None  {'version': '2.5.0.Final', 'connector': 'postg...  c  1754874771608   
1   None  {'version': '2.5.0.Final', 'connector': 'postg...  c  1754874771614   
2   None  {'version': '2.5.0.Final', 'connector': 'postg...  c  1754874771616   
3   None  {'version': '2.5.0.Final', 'connector': 'postg...  c  1754874771619   
4   None  {'version': '2.5.0.Final', 'connector': 'postg...  c  1754874771621   

  transaction  sk_id_curr name_contract_type code_gender flag_own_car  \
0        None      100001         Cash loans           F            N   
1        None      100005         Cash loans           M            N   
2        None      100013         Cash loans           M            Y   
3        None      100028         Cash loans           F            N   
4        None      100038         Cash loans           M            Y   

  flag_own_realty  ...  flag_document_18  flag_document_19  flag_document_

In [11]:
df_new

,before,source,op,ts_ms,transaction,sk_id_curr,name_contract_type,code_gender,flag_own_car,flag_own_realty,...,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874771608,None,100001,Cash loans,F,N,Y,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874771614,None,100005,Cash loans,M,N,Y,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874771616,None,100013,Cash loans,M,Y,Y,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874771619,None,100028,Cash loans,F,N,Y,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874771621,None,100038,Cash loans,M,Y,N,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874772267,None,106851,Cash loans,M,Y,Y,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,2.0
996,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874772267,None,106852,Cash loans,M,N,Y,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
997,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874772267,None,106853,Cash loans,M,Y,N,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
998,None,"{'version': '2.5.0.Final', 'connector': 'postg...",c,1754874772267,None,106854,Cash loans,F,N,Y,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,3.0
